In [ ]:
!pip install -q transformers==4.44.2 ijson tqdm scikit-learn

In [ ]:
import os, json, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import ijson
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATA_DIR = Path('/content/drive/.shortcut-targets-by-id/1YwiOUwtl8pCd2GD97Q_WEzwEUtSPoxFs/TwiBot-22')
assert (DATA_DIR / 'label.csv').exists(), f'label.csv not found in {DATA_DIR}'

WORK_DIR = Path('/content/twibot22_roberta')
WORK_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DATA_DIR =', DATA_DIR)
print('WORK_DIR =', WORK_DIR)
print('DEVICE   =', DEVICE)
print('Files    :', sorted(p.name for p in DATA_DIR.iterdir())[:20])

Mounted at /content/drive
DATA_DIR = /content/drive/.shortcut-targets-by-id/1YwiOUwtl8pCd2GD97Q_WEzwEUtSPoxFs/TwiBot-22
WORK_DIR = /content/twibot22_roberta
DEVICE   = cuda
Files    : ['edge.csv', 'hashtag.json', 'label.csv', 'list.json', 'readme.md', 'split.csv', 'tweet_0.json', 'tweet_1.json', 'tweet_2.json', 'tweet_3.json', 'tweet_4.json', 'tweet_5.json', 'tweet_6.json', 'tweet_7.json', 'tweet_8.json', 'user.json']


## 1. Labels + split (label_list.py equivalent)

In [ ]:
label_df = pd.read_csv(DATA_DIR / 'label.csv')
split_df = pd.read_csv(DATA_DIR / 'split.csv')

users_index_to_uid = list(label_df['id'])
uid_to_users_index = {u: i for i, u in enumerate(users_index_to_uid)}
N_USERS = len(users_index_to_uid)

# train.py convention: label = 1 - label, and str->int where human=1, bot=0
# so after the `1 - label` flip bots become the positive class (index 1)
raw_label = torch.tensor([1 if l == 'human' else 0 for l in label_df['label']], dtype=torch.long)
torch.save(raw_label, WORK_DIR / 'label_list.pt')

split_map = dict(zip(split_df['id'], split_df['split']))
split_idx = {'train': [], 'val': [], 'test': []}
for uid, idx in uid_to_users_index.items():
    s = split_map.get(uid)
    if s in split_idx:
        split_idx[s].append(idx)

print('users :', N_USERS)
print('train :', len(split_idx['train']))
print('val   :', len(split_idx['val']))
print('test  :', len(split_idx['test']))

users : 1000000
train : 700000
val   : 200000
test  : 100000


## 2. User descriptions (stream user.json)

In [ ]:
USER_JSON = DATA_DIR / 'user.json'
descriptions = [''] * N_USERS

with open(USER_JSON, 'rb') as f:
    for u in tqdm(ijson.items(f, 'item'), desc='user.json'):
        uid = u.get('id')
        idx = uid_to_users_index.get(uid)
        if idx is None:
            continue
        desc = u.get('description') or ''
        descriptions[idx] = desc

print('non-empty descriptions:', sum(1 for d in descriptions if d))

user.json: 0it [00:00, ?it/s]

non-empty descriptions: 875770


## 3. Per-user tweets (user_tweets_dict.py equivalent)

TwiBot-22 stores tweets across `tweet_0.json … tweet_8.json`. We build a `tid -> user_idx` map from `edge.csv` (`relation == 'post'`), then stream each tweet file and bucket texts by user.

In [ ]:
MAX_TWEETS_PER_USER = 20  # mean-pool cap; raise if RAM/GPU allows

edge_iter = pd.read_csv(DATA_DIR / 'edge.csv', chunksize=2_000_000)
tid_to_user = {}
for chunk in tqdm(edge_iter, desc='edge.csv'):
    posts = chunk[chunk['relation'] == 'post']
    for src, tgt in zip(posts['source_id'].values, posts['target_id'].values):
        uidx = uid_to_users_index.get(src)
        if uidx is None:
            continue
        if tid_to_user.get(tgt) is None:
            tid_to_user[tgt] = uidx

print('post edges kept:', len(tid_to_user))

edge.csv: 0it [00:00, ?it/s]

post edges kept: 88217457


In [ ]:
user_tweets = [[] for _ in range(N_USERS)]
tweet_files = sorted(DATA_DIR.glob('tweet_*.json'))
print('tweet files:', [p.name for p in tweet_files])

for tf in tweet_files:
    with open(tf, 'rb') as f:
        for t in tqdm(ijson.items(f, 'item'), desc=tf.name):
            tid = t.get('id')
            uidx = tid_to_user.get(tid)
            if uidx is None:
                continue
            if len(user_tweets[uidx]) >= MAX_TWEETS_PER_USER:
                continue
            text = t.get('text') or ''
            if text:
                user_tweets[uidx].append(text)

del tid_to_user; gc.collect()
covered = sum(1 for t in user_tweets if t)
print(f'users with >=1 tweet: {covered} / {N_USERS}')
np.save(WORK_DIR / 'user_tweets_dict.npy', np.array(user_tweets, dtype=object), allow_pickle=True)

tweet files: ['tweet_0.json', 'tweet_1.json', 'tweet_2.json', 'tweet_3.json', 'tweet_4.json', 'tweet_5.json', 'tweet_6.json', 'tweet_7.json', 'tweet_8.json']


tweet_0.json: 0it [00:00, ?it/s]

tweet_1.json: 0it [00:00, ?it/s]

tweet_2.json: 0it [00:00, ?it/s]

tweet_3.json: 0it [00:00, ?it/s]

...

tweet_5.json: 0it [00:00, ?it/s]

tweet_6.json: 0it [00:00, ?it/s]

tweet_7.json: 0it [00:00, ?it/s]

tweet_8.json: 0it [00:00, ?it/s]

users with >=1 tweet: 933872 / 1000000


## 4. RoBERTa feature tensors (tweets_tensor.py equivalent)

In [ ]:
from transformers import AutoTokenizer, AutoModel

ENCODER = 'roberta-base'
EMB_DIM = 768
TWEET_MAX_LEN = 50
DESC_MAX_LEN = 64
ENC_BATCH = 64

tok = AutoTokenizer.from_pretrained(ENCODER)
enc = AutoModel.from_pretrained(ENCODER).to(DEVICE).eval()
for p in enc.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def encode_batch(texts, max_len):
    batch = tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(DEVICE)
    out = enc(**batch).last_hidden_state
    mask = batch['attention_mask'].unsqueeze(-1).float()
    return ((out * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Description tensor
des_tensor = torch.zeros(N_USERS, EMB_DIM)
for i in tqdm(range(0, N_USERS, ENC_BATCH), desc='des_tensor'):
    chunk = [descriptions[j] if descriptions[j] else ' ' for j in range(i, min(i + ENC_BATCH, N_USERS))]
    des_tensor[i:i + len(chunk)] = encode_batch(chunk, DESC_MAX_LEN)

torch.save(des_tensor, WORK_DIR / 'des_tensor.pt')
print('des_tensor:', des_tensor.shape)

des_tensor:   0%|          | 0/15625 [00:00<?, ?it/s]

des_tensor: torch.Size([1000000, 768])


In [ ]:
# Tweet tensor: encode each user's tweets then mean-pool across tweets
tweets_tensor = torch.zeros(N_USERS, EMB_DIM)
for uidx in tqdm(range(N_USERS), desc='tweets_tensor'):
    texts = user_tweets[uidx]
    if not texts:
        tweets_tensor[uidx] = torch.randn(EMB_DIM) * 0.01
        continue
    vecs = []
    for i in range(0, len(texts), ENC_BATCH):
        vecs.append(encode_batch(texts[i:i + ENC_BATCH], TWEET_MAX_LEN))
    tweets_tensor[uidx] = torch.cat(vecs, 0).mean(0)

torch.save(tweets_tensor, WORK_DIR / 'tweets_tensor.pt')
print('tweets_tensor:', tweets_tensor.shape)

del enc, tok; torch.cuda.empty_cache(); gc.collect()

tweets_tensor:   0%|          | 0/1000000 [00:00<?, ?it/s]

tweets_tensor: torch.Size([1000000, 768])


182

## 5. Train MLPclassifier (train.py)

In [ ]:
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score

tweets_tensor = torch.load(WORK_DIR / 'tweets_tensor.pt')
des_tensor    = torch.load(WORK_DIR / 'des_tensor.pt')
label_all     = 1 - torch.load(WORK_DIR / 'label_list.pt')  # flip: bot=1

class Twibot22Dataset(Dataset):
    def __init__(self, name):
        idx = split_idx[name]
        self.tweet = tweets_tensor[idx]
        self.des   = des_tensor[idx]
        self.label = label_all[idx]
    def __len__(self): return len(self.tweet)
    def __getitem__(self, i): return self.tweet[i], self.des[i], self.label[i]

class MLPclassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, dropout=0.5):
        super().__init__()
        self.pre_model1 = nn.Linear(input_dim, input_dim // 2)
        self.pre_model2 = nn.Linear(input_dim, input_dim // 2)
        self.linear_relu_tweet = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LeakyReLU())
        self.dropout = nn.Dropout(p=dropout)
        self.classifier = nn.Linear(hidden_dim, 2)
    def forward(self, t, d):
        x = torch.cat((self.pre_model1(t), self.pre_model2(d)), dim=1)
        x = self.linear_relu_tweet(x)
        return self.classifier(self.dropout(x))

def report(preds_auc, preds, labels, tag):
    print(f'[{tag}] ACC={accuracy_score(labels, preds):.4f} '
          f'F1={f1_score(labels, preds):.4f} '
          f'ROC={roc_auc_score(labels, preds_auc):.4f} '
          f'P={precision_score(labels, preds):.4f} '
          f'R={recall_score(labels, preds):.4f}')

In [ ]:
BATCH_SIZE = 128
EPOCHS = 50
LR = 1e-3
WD = 1e-5

train_loader = DataLoader(Twibot22Dataset('train'), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(Twibot22Dataset('val'),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(Twibot22Dataset('test'),  batch_size=BATCH_SIZE)

model = MLPclassifier().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
loss_fn = nn.CrossEntropyLoss()

def run(loader, train=False):
    model.train(train)
    preds, aucs, labs, tot = [], [], [], 0.0
    for t, d, y in loader:
        t, d, y = t.to(DEVICE), d.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            out = model(t, d)
            loss = loss_fn(out, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * y.size(0)
        preds.append(out.argmax(-1).cpu().numpy())
        aucs.append(out[:, 1].detach().cpu().numpy())
        labs.append(y.cpu().numpy())
    return tot / len(loader.dataset), np.concatenate(aucs), np.concatenate(preds), np.concatenate(labs)

best_val = 0.0
for ep in range(EPOCHS):
    tr_loss, tr_auc, tr_p, tr_y = run(train_loader, train=True)
    va_loss, va_auc, va_p, va_y = run(val_loader)
    print(f'epoch {ep:02d}  train_loss={tr_loss:.4f}  val_loss={va_loss:.4f}')
    report(tr_auc, tr_p, tr_y, 'train')
    report(va_auc, va_p, va_y, 'val  ')
    val_acc = accuracy_score(va_y, va_p)
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), WORK_DIR / 'mlp_best.pt')

model.load_state_dict(torch.load(WORK_DIR / 'mlp_best.pt'))
te_loss, te_auc, te_p, te_y = run(test_loader)
print('\n=== best-val checkpoint on test ===')
report(te_auc, te_p, te_y, 'test ')

epoch 00  train_loss=0.1948  val_loss=0.6315
[train] ACC=0.9268 F1=0.2383 ROC=0.8509 P=0.6332 R=0.1468
[val  ] ACC=0.7219 F1=0.1158 ROC=0.7031 P=0.5213 R=0.0652
epoch 01  train_loss=0.1883  val_loss=0.7065
[train] ACC=0.9283 F1=0.2884 ROC=0.8633 P=0.6376 R=0.1863
[val  ] ACC=0.7229 F1=0.0714 ROC=0.7151 P=0.5657 R=0.0381
epoch 02  train_loss=0.1858  val_loss=0.7842
[train] ACC=0.9293 F1=0.3183 ROC=0.8683 P=0.6402 R=0.2118
[val  ] ACC=0.7228 F1=0.0748 ROC=0.7163 P=0.5578 R=0.0401
epoch 03  train_loss=0.1847  val_loss=0.7649
[train] ACC=0.9296 F1=0.3288 ROC=0.8712 P=0.6397 R=0.2213
[val  ] ACC=0.7250 F1=0.0597 ROC=0.7202 P=0.6756 R=0.0312
epoch 04  train_loss=0.1838  val_loss=0.6728
[train] ACC=0.9297 F1=0.3343 ROC=0.8725 P=0.6377 R=0.2266
[val  ] ACC=0.7308 F1=0.1827 ROC=0.7144 P=0.6038 R=0.1077
epoch 05  train_loss=0.1833  val_loss=0.6644
[train] ACC=0.9299 F1=0.3398 ROC=0.8729 P=0.6389 R=0.2314
[val  ] ACC=0.7342 F1=0.2017 ROC=0.7180 P=0.6294 R=0.1201
epoch 06  train_loss=0.1829  val_l

## 6. Test statistics, confidence, and sample cases

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
test_ds = Twibot22Dataset('test')
test_idx = np.array(split_idx['test'])

probs_all, preds_all, labels_all = [], [], []
with torch.no_grad():
    for t, d, y in DataLoader(test_ds, batch_size=256):
        logits = model(t.to(DEVICE), d.to(DEVICE))
        p = F.softmax(logits, dim=-1).cpu().numpy()
        probs_all.append(p)
        preds_all.append(p.argmax(1))
        labels_all.append(y.numpy())
probs_all  = np.concatenate(probs_all)     # [N, 2]  col 1 = bot
preds_all  = np.concatenate(preds_all)
labels_all = np.concatenate(labels_all)
confidences = probs_all.max(axis=1)
bot_scores  = probs_all[:, 1]

In [ ]:
acc  = accuracy_score(labels_all, preds_all)
f1   = f1_score(labels_all, preds_all)
roc  = roc_auc_score(labels_all, bot_scores)
prec = precision_score(labels_all, preds_all)
rec  = recall_score(labels_all, preds_all)

print(f'Test set size : {len(labels_all)}')
print(f'Accuracy      : {acc:.4f}')
print(f'F1 (bot)      : {f1:.4f}')
print(f'ROC-AUC       : {roc:.4f}')
print(f'Precision     : {prec:.4f}')
print(f'Recall        : {rec:.4f}')

cm = confusion_matrix(labels_all, preds_all)
print('\nConfusion matrix (rows=true, cols=pred)')
print('              pred_human  pred_bot')
print(f'true_human    {cm[0,0]:>10d}  {cm[0,1]:>8d}')
print(f'true_bot      {cm[1,0]:>10d}  {cm[1,1]:>8d}')

print('\nClassification report')
print(classification_report(labels_all, preds_all, target_names=['human', 'bot'], digits=4))

Test set size : 100000
Accuracy      : 0.7512
F1 (bot)      : 0.4034
ROC-AUC       : 0.7532
Precision     : 0.6863
Recall        : 0.2856

Confusion matrix (rows=true, cols=pred)
              pred_human  pred_bot
true_human         66711      3845
true_bot           21034      8410

Classification report
              precision    recall  f1-score   support

       human     0.7603    0.9455    0.8428     70556
         bot     0.6863    0.2856    0.4034     29444

    accuracy                         0.7512    100000
   macro avg     0.7233    0.6156    0.6231    100000
weighted avg     0.7385    0.7512    0.7134    100000



In [ ]:
print(f'Mean   confidence : {confidences.mean():.4f}')
print(f'Median confidence : {np.median(confidences):.4f}')
print(f'Std    confidence : {confidences.std():.4f}')
print(f'Min / Max         : {confidences.min():.4f} / {confidences.max():.4f}')

correct = preds_all == labels_all
print(f'\nMean conf when correct : {confidences[correct].mean():.4f}')
print(f'Mean conf when wrong   : {confidences[~correct].mean():.4f}')

buckets = [(0.50, 0.60), (0.60, 0.70), (0.70, 0.80), (0.80, 0.90), (0.90, 0.95), (0.95, 1.01)]
print('\nconfidence bucket   count   accuracy')
for lo, hi in buckets:
    m = (confidences >= lo) & (confidences < hi)
    if m.sum() == 0:
        print(f'  [{lo:.2f}, {hi:.2f})        0       -')
    else:
        print(f'  [{lo:.2f}, {hi:.2f})   {m.sum():>6d}    {correct[m].mean():.4f}')

high = confidences >= 0.90
print(f'\nHigh-confidence (>=0.90) coverage : {high.mean():.2%}')
print(f'High-confidence accuracy          : {correct[high].mean():.4f}')

Mean   confidence : 0.8538
Median confidence : 0.9361
Std    confidence : 0.1601
Min / Max         : 0.5000 / 0.9998

Mean conf when correct : 0.8770
Mean conf when wrong   : 0.7839

confidence bucket   count   accuracy
  [0.50, 0.60)    14164    0.6015
  [0.60, 0.70)     7250    0.5637
  [0.70, 0.80)     7731    0.6072
  [0.80, 0.90)    12398    0.6920
  [0.90, 0.95)    13493    0.7728
  [0.95, 1.01)    44964    0.8632

High-confidence (>=0.90) coverage : 58.46%
High-confidence accuracy          : 0.8423


In [ ]:
def show_case(pos, tag):
    uidx = test_idx[pos]
    uid  = users_index_to_uid[uidx]
    true = 'bot' if labels_all[pos] == 1 else 'human'
    pred = 'bot' if preds_all[pos]  == 1 else 'human'
    desc = (descriptions[uidx] or '').replace('\n', ' ')[:120]
    tw   = user_tweets[uidx][0].replace('\n', ' ')[:140] if user_tweets[uidx] else '(no tweets)'
    print(f'[{tag}] uid={uid}  true={true}  pred={pred}  P(bot)={bot_scores[pos]:.3f}')
    print(f'    desc : {desc}')
    print(f'    tweet: {tw}\n')

order = np.argsort(-confidences)

print('=== Top-5 most-confident correct BOT predictions ===')
shown = 0
for pos in order:
    if shown >= 5: break
    if labels_all[pos] == 1 and preds_all[pos] == 1:
        show_case(pos, 'correct bot'); shown += 1

print('=== Top-5 most-confident correct HUMAN predictions ===')
shown = 0
for pos in order:
    if shown >= 5: break
    if labels_all[pos] == 0 and preds_all[pos] == 0:
        show_case(pos, 'correct human'); shown += 1

print('=== Top-5 most-confident MISCLASSIFICATIONS ===')
shown = 0
for pos in order:
    if shown >= 5: break
    if preds_all[pos] != labels_all[pos]:
        show_case(pos, 'wrong'); shown += 1

print('=== 5 lowest-confidence predictions (model unsure) ===')
for pos in np.argsort(confidences)[:5]:
    show_case(pos, 'unsure')

=== Top-5 most-confident correct BOT predictions ===
[correct bot] uid=u1038802275710459904  true=bot  pred=bot  P(bot)=0.998
    desc : 22 Manchester 📈 📉 🌏
    tweet: #roleplay #rp #nsfwroleplay #ddgl #kink #hardkink #horny #tight #virgin #tightteen #cum #hardfuck #mommy #ageplaykink #24hoursinpolicecustod

[correct bot] uid=u993546106930843648  true=bot  pred=bot  P(bot)=0.998
    desc : CEO Nkarta
    tweet: Biotech Leaders Look to Halt Business Deals in Russia https://t.co/Eo5TTjVOlN via @wsj

[correct bot] uid=u3095444538  true=bot  pred=bot  P(bot)=0.988
    desc : 
    tweet: #amateurgaysex #tatoo #gay #instagay #gayboy #cute #love #twinks #picoftheday #boy #gayman #hotboys #sexymen #abs #instagood #gayguy #gayhot

[correct bot] uid=u1324425394242596865  true=bot  pred=bot  P(bot)=0.988
    desc : We sell pet products. Our new product, Safer Scooper, is coming soon.
    tweet: #dogs #dogsofinstagram #dog #dogstagram #puppy #instadog #doglover #dogoftheday #pets #doglovers #love 

In [ ]:
out_df = pd.DataFrame({
    'user_id'   : [users_index_to_uid[i] for i in test_idx],
    'true'      : ['bot' if y == 1 else 'human' for y in labels_all],
    'pred'      : ['bot' if y == 1 else 'human' for y in preds_all],
    'p_bot'     : bot_scores,
    'confidence': confidences,
    'correct'   : correct,
})
out_df.to_csv(WORK_DIR / 'test_predictions.csv', index=False)
print('Saved', WORK_DIR / 'test_predictions.csv', 'shape=', out_df.shape)
out_df.head()

Saved /content/twibot22_roberta/test_predictions.csv shape= (100000, 6)


,user_id,true,pred,p_bot,confidence,correct
0,u1217628182611927040,human,human,0.022746,0.977255,True
1,u1266703520205549568,human,human,0.126516,0.873483,True
2,u1365527332627247104,bot,human,0.172453,0.827547,False
3,u1341789703633178624,bot,human,0.431655,0.568345,False
4,u2465283662,bot,bot,0.867977,0.867977,True


In [ ]:
# Class balance + prediction distribution
from collections import Counter

n = len(labels_all)
true_counts = Counter(labels_all.tolist())
pred_counts = Counter(preds_all.tolist())

print('=== Label / prediction distribution ===')
print(f'Test size            : {n}')
print(f'True   human / bot   : {true_counts[0]} ({true_counts[0]/n:.2%})  /  {true_counts[1]} ({true_counts[1]/n:.2%})')
print(f'Pred   human / bot   : {pred_counts[0]} ({pred_counts[0]/n:.2%})  /  {pred_counts[1]} ({pred_counts[1]/n:.2%})')

tn, fp, fn_, tp = cm.ravel()
print('\n=== Raw error counts ===')
print(f'TP (bot caught)      : {tp}')
print(f'TN (human passed)    : {tn}')
print(f'FP (human->bot)      : {fp}')
print(f'FN (bot missed)      : {fn_}')

spec = tn / (tn + fp) if (tn + fp) else 0.0
bal_acc = 0.5 * (rec + spec)
mcc_num = tp * tn - fp * fn_
mcc_den = np.sqrt((tp + fp) * (tp + fn_) * (tn + fp) * (tn + fn_))
mcc = mcc_num / mcc_den if mcc_den else 0.0

print('\n=== Aggregate metrics ===')
print(f'Specificity (TNR)    : {spec:.4f}')
print(f'Balanced accuracy    : {bal_acc:.4f}')
print(f'Matthews corrcoef    : {mcc:.4f}')

=== Label / prediction distribution ===
Test size            : 100000
True   human / bot   : 70556 (70.56%)  /  29444 (29.44%)
Pred   human / bot   : 87745 (87.74%)  /  12255 (12.26%)

=== Raw error counts ===
TP (bot caught)      : 8410
TN (human passed)    : 66711
FP (human->bot)      : 3845
FN (bot missed)      : 21034

=== Aggregate metrics ===
Specificity (TNR)    : 0.9455
Balanced accuracy    : 0.6156
Matthews corrcoef    : 0.3213


In [ ]:
# Threshold sweep on P(bot)
from sklearn.metrics import precision_recall_curve, average_precision_score, log_loss, brier_score_loss

print('=== Threshold sweep (P(bot) >= thr) ===')
print('thr     ACC      F1       P        R        FPR')
for thr in [0.30, 0.40, 0.50, 0.60, 0.70]:
    p = (bot_scores >= thr).astype(int)
    tn_, fp_, fn2, tp_ = confusion_matrix(labels_all, p).ravel()
    a  = (tp_ + tn_) / n
    pr = tp_ / (tp_ + fp_) if (tp_ + fp_) else 0.0
    rc = tp_ / (tp_ + fn2) if (tp_ + fn2) else 0.0
    f  = 2 * pr * rc / (pr + rc) if (pr + rc) else 0.0
    fpr = fp_ / (fp_ + tn_) if (fp_ + tn_) else 0.0
    print(f'{thr:.2f}   {a:.4f}  {f:.4f}  {pr:.4f}  {rc:.4f}  {fpr:.4f}')

prec_c, rec_c, _ = precision_recall_curve(labels_all, bot_scores)
ap = average_precision_score(labels_all, bot_scores)
print(f'\nAverage precision (AP): {ap:.4f}')
print(f'Log loss              : {log_loss(labels_all, probs_all):.4f}')
print(f'Brier score           : {brier_score_loss(labels_all, bot_scores):.4f}')

=== Threshold sweep (P(bot) >= thr) ===
thr     ACC      F1       P        R        FPR
0.30   0.7550  0.5306  0.6088  0.4702  0.1261
0.40   0.7578  0.4834  0.6498  0.3849  0.0866
0.50   0.7512  0.4034  0.6863  0.2856  0.0545
0.60   0.7158  0.1315  0.6569  0.0731  0.0159
0.70   0.7094  0.0556  0.6438  0.0290  0.0067

Average precision (AP): 0.5525
Log loss              : 0.6264
Brier score           : 0.1882


In [ ]:
# Expected Calibration Error (ECE) + per-class confidence
n_bins = 10
bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
ece = 0.0
print('=== Calibration bins (conf vs. accuracy) ===')
print('bin            n     avg_conf   accuracy   gap')
for i in range(n_bins):
    lo, hi = bin_edges[i], bin_edges[i + 1]
    m = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
    if m.sum() == 0:
        continue
    avg_conf = confidences[m].mean()
    acc_bin  = correct[m].mean()
    gap      = abs(avg_conf - acc_bin)
    ece     += (m.sum() / n) * gap
    print(f'({lo:.2f},{hi:.2f}]  {m.sum():>5d}   {avg_conf:.4f}    {acc_bin:.4f}    {gap:.4f}')
print(f'\nECE (10 bins)        : {ece:.4f}')

print('\n=== Per-class confidence ===')
for cls, name in [(0, 'human'), (1, 'bot')]:
    m = labels_all == cls
    print(f'{name:6s}  n={m.sum():>5d}  mean_conf={confidences[m].mean():.4f}  recall={correct[m].mean():.4f}')

=== Calibration bins (conf vs. accuracy) ===
bin            n     avg_conf   accuracy   gap
(0.50,0.60]  14164   0.5443    0.6015    0.0572
(0.60,0.70]   7250   0.6492    0.5637    0.0855
(0.70,0.80]   7731   0.7518    0.6072    0.1446
(0.80,0.90]  12398   0.8555    0.6920    0.1635
(0.90,1.00]  58457   0.9674    0.8423    0.1250

ECE (10 bins)        : 0.1188

=== Per-class confidence ===
human   n=70556  mean_conf=0.8966  recall=0.9455
bot     n=29444  mean_conf=0.7515  recall=0.2856


In [ ]:
# Bootstrap 95% CIs on headline metrics
rng = np.random.default_rng(0)
B = 1000
boot_acc, boot_f1, boot_roc = [], [], []
idx_arr = np.arange(n)
for _ in range(B):
    s = rng.choice(idx_arr, size=n, replace=True)
    if len(np.unique(labels_all[s])) < 2:
        continue
    boot_acc.append(accuracy_score(labels_all[s], preds_all[s]))
    boot_f1.append(f1_score(labels_all[s], preds_all[s]))
    boot_roc.append(roc_auc_score(labels_all[s], bot_scores[s]))

def ci(xs):
    return np.percentile(xs, 2.5), np.percentile(xs, 97.5)

print(f'=== Bootstrap 95% CIs (B={B}) ===')
lo, hi = ci(boot_acc); print(f'Accuracy : {acc:.4f}  [{lo:.4f}, {hi:.4f}]')
lo, hi = ci(boot_f1);  print(f'F1 (bot) : {f1:.4f}  [{lo:.4f}, {hi:.4f}]')
lo, hi = ci(boot_roc); print(f'ROC-AUC  : {roc:.4f}  [{lo:.4f}, {hi:.4f}]')

=== Bootstrap 95% CIs (B=1000) ===
Accuracy : 0.7512  [0.7486, 0.7540]
F1 (bot) : 0.4034  [0.3972, 0.4095]
ROC-AUC  : 0.7532  [0.7500, 0.7566]
